# Error Detection 
The objective of this phase is to conduct a comprehensive analysis of the discrepancies identified during the Comparison phase. Rather than simply knowing that errors exist, we now investigate WHY they occurred, categorize them by business impact, and prioritize resolution efforts.

After comparison revealed £17,857.56 in financial variance across 1,980 discrepant records, this phase focuses on transforming raw discrepancy data into actionable business intelligence.

This step includes:

- Error Categorization & Classification:
    - Systematic grouping of discrepancies by type (data quality, system sync, business logic)
    - Financial impact scoring (High/Medium/Low risk categories)
    - Frequency analysis to identify the most common error patterns

- Root Cause Analysis:
    - Investigation of underlying causes for each error category
    - Pattern detection across time periods, products, and customer segments  
    - Correlation analysis between different types of errors

- Business Impact Assessment:
    - Quantification of operational and financial risks
    - Assessment of compliance and audit implications
    - Evaluation of customer experience impact

- Trend & Pattern Analysis:
    - Temporal distribution of errors (seasonal patterns, system events)
    - Product/customer segment analysis (which areas are most affected)
    - Error concentration analysis (are problems systemic or isolated?)

- Prioritization Framework:
    - Development of error priority matrix based on impact vs. effort to resolve
    - Risk scoring methodology for different error types
    - Recommendations for immediate vs. long-term remediation

- Validation Effectiveness Review:
    - Analysis of how well the Validation phase detected different error types
    - Identification of gaps in current validation processes
    - Recommendations for improving future error detection

## Purpose: 
Transform discrepancy identification into strategic remediation planning. This analysis provides the foundation for targeted resolution efforts and process improvements, ensuring that the most critical issues are addressed first while building systematic defenses against future data quality problems.

**Step 1** Error Categorization and Classification

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import ast

def load_data(file_path):
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"File {file_path} loaded correctly using latin1 encoding. Loaded {len(df)} rows.")
    except Exception:
        df = pd.read_csv(file_path, encoding='cp1252')
        print(f"File {file_path} loaded correctly using cp1252 encoding. Loaded {len(df)} rows.")
    
    # parse validation flags
    df['validation_flags'] = df['validation_flags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else []) # parsing strings in the validation_flags column into actual lists
    return df

df_oms = load_data('validation/oms_final_clean.csv')
df_wms = load_data('validation/wms_final_clean.csv')


File validation/oms_final_clean.csv loaded correctly using latin1 encoding. Loaded 541909 rows.
File validation/wms_final_clean.csv loaded correctly using latin1 encoding. Loaded 542409 rows.


In [17]:
print("STEP 1: Aggregating duplicates for consistent analysis")

# aggregation logic same as in comparison phase 
agg_logic_oms = {
    'UnitPrice': 'max',
    'InvoiceDate': 'first',
    'CustomerID': 'first',
    'Country': 'first',
    'Description': 'first',
    'validation_status': 'first',
    'validation_flags': lambda x: list(set([item for sublist in x for item in sublist]))
}

# define agg logic for wms
agg_logic_wms = agg_logic_oms.copy()

# apply aggregation
df_oms_agg = df_oms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg(agg_logic_oms).reset_index()
df_wms_agg = df_wms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg(agg_logic_wms).reset_index()

print(f"OMS after aggregation: {len(df_oms_agg)} unique business transactions.")
print(f"WMS after aggregation: {len(df_wms_agg)} unique business transactions.")

STEP 1: Aggregating duplicates for consistent analysis
OMS after aggregation: 536478 unique business transactions.
WMS after aggregation: 536478 unique business transactions.
